<p align="center"><b>Assignment 10</b></p>

 Write a program to denoise the input image using adaptive median filter.
Input the initial window size and maximum size of window (Smax) from
the user. Compare the performance of median filter and adaptive median
filter, using the PSNR values.

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('input.jpg', 0)

if img is None:
    print("Image not found")
    exit()

img = img.astype(float)

Smin = int(input("Enter initial window size: "))
Smax = int(input("Enter maximum window size: "))

noisy = img.copy()
prob = 0.1

random = np.random.random(img.shape)
noisy[random < prob/2] = 0
noisy[random > 1-prob/2] = 255


def median_filter(image, size):
    pad = size // 2
    padded = np.pad(image, pad, mode='reflect')
    result = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = padded[i:i+size, j:j+size]
            result[i, j] = np.median(window)

    return result


def adaptive_median(image, Smin, Smax):
    pad = Smax // 2
    padded = np.pad(image, pad, mode='reflect')
    result = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):

            size = Smin
            pixel = image[i, j]

            while size <= Smax:
                p = pad
                start = p - size//2

                window = padded[
                    i+start:i+start+size,
                    j+start:j+start+size
                ]

                zmin = np.min(window)
                zmax = np.max(window)
                zmed = np.median(window)

                A1 = zmed - zmin
                A2 = zmed - zmax

                if A1 > 0 and A2 < 0:
                    B1 = pixel - zmin
                    B2 = pixel - zmax

                    if B1 > 0 and B2 < 0:
                        result[i, j] = pixel
                    else:
                        result[i, j] = zmed

                    break

                else:
                    size += 2

            else:
                result[i, j] = zmed

    return result


def calculate_psnr(original, filtered):
    mse = np.mean((original - filtered) ** 2)

    if mse == 0:
        return float('inf')

    return 10 * np.log10((255 ** 2) / mse)


median_img = median_filter(noisy, Smin)
adaptive_img = adaptive_median(noisy, Smin, Smax)

psnr_median = calculate_psnr(img, median_img)
psnr_adaptive = calculate_psnr(img, adaptive_img)

print("\nPSNR Comparison")
print("Median Filter PSNR:", psnr_median, "dB")
print("Adaptive Median PSNR:", psnr_adaptive, "dB")

images = [img, noisy, median_img, adaptive_img]
titles = ['Original Image', 'Noisy Image',
          'Median Filter', 'Adaptive Median Filter']

plt.figure(figsize=(10, 8))

for i in range(4):
    plt.subplot(2, 2, i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()


Write a program to threshold the image using Otsu's global threshold for
segmentation of input gray scale image

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('input.jpg', 0)

if img is None:
    print("Image not found")
    exit()

hist, bins = np.histogram(img.flatten(), 256, [0, 256])

total = img.size
sum_total = np.sum(np.arange(256) * hist)

sum_bg = 0
weight_bg = 0
max_variance = 0
threshold = 0

for t in range(256):
    weight_bg += hist[t]

    if weight_bg == 0:
        continue

    weight_fg = total - weight_bg

    if weight_fg == 0:
        break

    sum_bg += t * hist[t]

    mean_bg = sum_bg / weight_bg
    mean_fg = (sum_total - sum_bg) / weight_fg

    variance = weight_bg * weight_fg * (mean_bg - mean_fg) ** 2

    if variance > max_variance:
        max_variance = variance
        threshold = t

print("Optimal Threshold:", threshold)

segmented = np.zeros_like(img)
segmented[img > threshold] = 255

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(segmented, cmap='gray')
plt.title("Otsu Segmentation")
plt.axis('off')

plt.tight_layout()
plt.show()
